<a href="https://colab.research.google.com/github/crypt0d1v3r/CannyValley/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies for pulling datasets
%pip install kagglehub
%pip install datasets
%pip install torch
%pip install torchvision
%pip install numpy
%pip install matplotlib


   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   ---------------------------------------- 0/2 [kagglesdk]
   -------------------- ------------------- 1/2 [kagglehub]
   -------------------- ------------------- 1/2 [kagglehub]
   ---------------------------------------- 2/2 [kagglehub]

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/536.6 kB ? eta -:--:--
   ---------------------------------------- 536.6/536.6 kB 8.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 24.8 MB/s  0:00:00

   ------ --------------------------------- 1/6 [multiprocess]
   ------ --------------------------------- 1/6 [multiprocess]
   -------------------- ------------------- 3/6 [typer-slim]
   -------------------------- ------------- 4/6 [huggingface-hub]
   ------------------------

In [ ]:
import kagglehub
from datasets import load_dataset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch
import os
import numpy as np
import matplotlib.pyplot as plt

class CNN(torch.nn.Module):
  def __init__(self):
      super().__init__()
      self.model = torch.nn.Sequential(
          torch.nn.Conv2d(in_channels=3, out_channels=32,
                          kernel_size=3, padding=1),
          torch.nn.ReLU(),
          torch.nn.MaxPool2d(kernel_size=2),
          torch.nn.Conv2d(in_channels=32, out_channels=64,
                          kernel_size=3, padding=1),
          torch.nn.ReLU(),
          torch.nn.MaxPool2d(kernel_size=2),
          torch.nn.Conv2d(in_channels=64, out_channels=64,
                          kernel_size=3, padding=1),
          torch.nn.ReLU(),
          torch.nn.MaxPool2d(kernel_size=2),
          torch.nn.Flatten(),
          torch.nn.Linear(64 * 28 * 28, 512),
          torch.nn.ReLU(),
          torch.nn.Linear(512, 2)
      )

  def forward(self, x):
      return self.model(x)

100%|██████████| 105M/105M [00:03<00:00, 33.1MB/s] 

Extracting files...


In [ ]:
dataset_path = "./datasets"
dataset_source = "birdy654/cifake-real-and-ai-generated-synthetic-images"
if dataset_source == "birdy654/cifake-real-and-ai-generated-synthetic-images":
  dataset_path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
elif dataset_source == "Hemg/AI-vs-Real-images":
  dataset_path = load_dataset("Hemg/AI-vs-Real-images")
elif dataset_source == "bitmind/AI-vs-Real-Dataset-Images-Proper":
  dataset_path = load_dataset("bitmind/AI-vs-Real-Dataset-Images-Proper")
else:
  raise Exception(f"Dataset Source {dataset_source} Unknown")

Can choose from the following datasets

---



*   birdy654/cifake-real-and-ai-generated-synthetic-images
*   Hemg/AI-vs-Real-images
*   bitmind/AI-vs-Real-Dataset-Images-Proper



In [ ]:
data_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dir = os.path.join(dataset_path, 'train')
test_dir = os.path.join(dataset_path , 'test')

train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Get class names
class_names = train_dataset.classes
print(f"Classes: {class_names}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model = CNN().to(device)

In [ ]:
num_epochs = 50
learning_rate = 0.001
weight_decay = 0.01
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=learning_rate, weight_decay=weight_decay)

train_loss_list = []
for epoch in range(num_epochs):
    print(f'Epoch {epoch+1}/{num_epochs}:', end=' ')
    train_loss = 0
    model.train()
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss_list.append(train_loss / len(train_loader))
    print(f"Training loss = {train_loss_list[-1]}")

plt.plot(range(1, num_epochs + 1), train_loss_list)
plt.xlabel("Number of epochs")
plt.ylabel("Training loss")
plt.show()

test_acc = 0
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        y_true = labels.to(device)
        outputs = model(images)
        _, y_pred = torch.max(outputs.data, 1)
        test_acc += (y_pred == y_true).sum().item()

print(f"Test set accuracy = {100 * test_acc / len(test_dataset)} %")

num_images = 5
y_true_name = [class_names[y_true[idx]] for idx in range(num_images)]
y_pred_name = [class_names[y_pred[idx]] for idx in range(num_images)]
title = f"Actual labels: {y_true_name}, Predicted labels: {y_pred_name}"

plt.imshow(np.transpose(torchvision.utils.make_grid(images[:num_images].cpu(), normalize=True, padding=1).numpy(), (1, 2, 0)))
plt.title(title)
plt.axis("off")
plt.show()
  